In [ ]:
# 导入所需库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score
from sklearn.multioutput import MultiOutputRegressor
import warnings
warnings.filterwarnings("ignore")

# 尝试导入TabPFNRegressor
from tabpfn import TabPFNRegressor

# 设置绘图样式
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False

# 1. 加载数据
print("正在加载数据...")
df_original = pd.read_excel('../data\quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# 6. 使用MultiOutputRegressor包装TabPFN进行多输出回归
print("训练TabPFN多输出回归模型...")

# 创建基本的TabPFN回归模型
base_model = TabPFNRegressor()

# 使用MultiOutputRegressor包装模型以处理多个目标
model = MultiOutputRegressor(base_model)

# 训练模型
model.fit(X_train_scaled, y_train)

# 预测
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# 计算整体指标
train_mse = mean_squared_error(y_train, y_pred_train)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_pred_train)
train_r2 = r2_score(y_train, y_pred_train)
train_explained_var = explained_variance_score(y_train, y_pred_train)

test_mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)
test_explained_var = explained_variance_score(y_test, y_pred_test)

# print("\n整体模型评价:")
# print(f"训练集评价:")
# print(f"  MSE: {train_mse:.4f}, RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}")
# print(f"  R2: {train_r2:.4f}, Explained Variance: {train_explained_var:.4f}")

# print(f"测试集评价:")
# print(f"  MSE: {test_mse:.4f}, RMSE: {test_rmse:.4f}, MAE: {test_mae:.4f}")
# print(f"  R2: {test_r2:.4f}, Explained Variance: {test_explained_var:.4f}")
# print("-" * 50)

# 为每个目标变量单独计算指标
print("\n各目标变量的评价:")
# 创建一个DataFrame用于存储各目标变量的R²值
r2_results = pd.DataFrame(index=target_vars, columns=['训练集R²', '测试集R²'])

for i, target in enumerate(target_vars):
    target_train_r2 = r2_score(y_train[target], y_pred_train[:, i])
    target_test_r2 = r2_score(y_test[target], y_pred_test[:, i])
    
    print(f"{target}:")
    print(f"  训练集R²: {target_train_r2:.4f}")
    print(f"  测试集R²: {target_test_r2:.4f}")
    
    # 存储到DataFrame中
    r2_results.loc[target] = [target_train_r2, target_test_r2]

# 计算平均R²值
mean_train_r2 = r2_results['训练集R²'].mean()
mean_test_r2 = r2_results['测试集R²'].mean()

print("\n平均R²指标汇总:")
print(f"所有目标变量的平均训练集R²: {mean_train_r2:.4f}")
print(f"所有目标变量的平均测试集R²: {mean_test_r2:.4f}")